# 실험 제목
- 담당: 김영빈
- 날짜: 26/09/25
- 목적: RAG 문서를 임베딩해서 pgvector에 저장하고, 질문으로 top-k 검색까지 확인 (1단계: 임베딩 모델 동작 확인)

> 끝나면 결과를 `experiments/LOG.md`에 한 줄 남기기

## 1단계. 임베딩 모델 로드

`dragonkue/snowflake-arctic-embed-l-v2.0-ko`를 sentence-transformers로 불러온다.

이 모델은 **질문(query)과 문서(document)를 다르게 인코딩해야 하는 비대칭(asymmetric) 모델**이다:
- 질문: `model.encode(text, prompt_name="query")` — 검색에 최적화된 특수 프롬프트가 내부적으로 붙음
- 문서: `model.encode(text)` — 접두사 없이 그대로

이 차이를 지키지 않으면 같은 모델을 쓰고도 검색 품질이 떨어진다(모델 카드에 명시된 내용).
처음 실행하면 허깅페이스에서 모델 가중치를 다운로드하므로 몇 분 걸릴 수 있다.

In [ ]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "dragonkue/snowflake-arctic-embed-l-v2.0-ko"

print("모델 로드 중...")
model = SentenceTransformer(EMBEDDING_MODEL)
print("완료. 임베딩 차원:", model.get_embedding_dimension())

## 2단계. 질문 vs 문서 인코딩이 실제로 다른지, 결과가 그럴듯한지 확인

같은 주제(대출 만기 연장)의 질문 1개와 관련 있는 문서 1개, 그리고 관련 없는 문서(보험금 청구) 1개를 인코딩해서
코사인 유사도를 비교한다. **관련 있는 문서 쪽 유사도가 더 높게 나와야** 모델이 제대로 동작하는 것이다.

`normalize_embeddings=True`로 벡터를 정규화하면 내적(dot product)이 곧 코사인 유사도가 된다
(모델 카드에서 권장하는 방식, pgvector의 `vector_cosine_ops`와도 맞는 방식).

In [ ]:
import numpy as np

sample_query = "대출 만기를 연장하려면 어떻게 해야 하나요?"
sample_doc_relevant = (
    "[요구사항] 대출 만기 연장 절차를 안내하시오.\n"
    "[고객 질문] 대출 만기가 다가오는데 연장하려면 어떻게 해야 하나요?"
)
sample_doc_irrelevant = (
    "[요구사항] 보험금 청구 절차를 안내하시오.\n"
    "[고객 질문] 자동차 사고가 났는데 보험금 청구는 어떻게 하나요?"
)

# 질문은 prompt_name="query" 를 붙여서 인코딩 (검색용 특수 프롬프트)
query_emb = model.encode(sample_query, prompt_name="query", normalize_embeddings=True)
# 문서는 접두사 없이 그대로 인코딩
doc_relevant_emb = model.encode(sample_doc_relevant, normalize_embeddings=True)
doc_irrelevant_emb = model.encode(sample_doc_irrelevant, normalize_embeddings=True)

def cosine(a, b):
    return float(np.dot(a, b))  # 정규화된 벡터라 내적 = 코사인 유사도

print(f"질문 벡터 차원: {query_emb.shape}")
print(f"[관련 있음] 대출 연장 문서와 유사도: {cosine(query_emb, doc_relevant_emb):.4f}")
print(f"[관련 없음] 보험금 청구 문서와 유사도: {cosine(query_emb, doc_irrelevant_emb):.4f}")

## 3단계. pgvector 연결 + `document_chunk` 테이블 생성

DB에 아래 구조의 테이블을 만든다. `text`(임베딩 대상 원문)와 `embedding`(벡터)을 같은 행에 둬서,
검색 결과에서 바로 원문을 같이 받을 수 있게 한다. 메타데이터는 회의에서 합의한 4개만 컬럼으로 둔다.

```
document_chunk
├─ doc_id              TEXT PRIMARY KEY   원본 QA ID. 중복 방지 + 재실행 시 upsert 기준
├─ text                TEXT               요구사항+질문+답변+꼬리질문+종합답변을 합친 임베딩 대상 원문
├─ category             TEXT               은행 / 보험 / 증권
├─ consulting_topic     TEXT               세부 상담 주제
├─ qa_topic             TEXT               QA 단위 주제
├─ consulting_purpose   TEXT               상담 목적
└─ embedding            VECTOR(1024)       임베딩 벡터 (1단계에서 확인한 모델 출력 차원)
```

인덱스(HNSW)는 아직 안 만든다 — 데이터를 다 넣은 뒤에 만들어야 빠르다
(먼저 만들면 문서를 넣을 때마다 검색 그래프를 다시 계산해서 느려진다). 이건 5단계에서 한다.

In [ ]:
import os

import psycopg
from dotenv import load_dotenv
from pgvector.psycopg import register_vector

load_dotenv()  # .env 에서 DATABASE_URL 읽기 (docker-compose 로 띄운 로컬 pgvector)

EMBEDDING_DIM = model.get_embedding_dimension()  # 1단계에서 확인한 1024

conn = psycopg.connect(os.environ["DATABASE_URL"], autocommit=True)

with conn.cursor() as cur:
    cur.execute("CREATE EXTENSION IF NOT EXISTS vector")  # pgvector 확장 먼저 활성화 (이미 있으면 아무 일도 안 함)

register_vector(conn)  # 확장이 있어야 psycopg 가 vector 타입을 인식할 수 있음 (그래서 extension 생성 다음에 호출)

with conn.cursor() as cur:
    cur.execute(f"""
        CREATE TABLE IF NOT EXISTS document_chunk (
            doc_id TEXT PRIMARY KEY,
            text TEXT NOT NULL,
            category TEXT,
            consulting_topic TEXT,
            qa_topic TEXT,
            consulting_purpose TEXT,
            embedding VECTOR({EMBEDDING_DIM})
        )
    """)

print("테이블 생성 완료 (또는 이미 존재)")

## 4단계. 테이블 생성 확인

DB에 실제로 어떤 컬럼이 만들어졌는지 조회해서 눈으로 확인한다 (DBeaver를 새로고침해서 봐도 동일하게 보인다).

In [ ]:
with conn.cursor() as cur:
    cur.execute("""
        SELECT column_name, data_type
        FROM information_schema.columns
        WHERE table_name = \'document_chunk\'
        ORDER BY ordinal_position
    """)
    for column_name, data_type in cur.fetchall():
        print(f"  {column_name:<20} {data_type}")

## 5단계. 전체 문서 임베딩 + 적재 (재개 가능하게)

⚠️ 처음 전체(8만 건)를 한 번에 돌리다가 macOS가 멈춰서 강제 종료한 적이 있다. 그래서 이 버전은:
- **이미 `document_chunk`에 들어간 `doc_id`는 건너뛴다** — 중간에 멈춰도 이어서 실행하면 중복 없이 재개된다.
- 배치마다 `torch.mps.empty_cache()`로 GPU 메모리를 정리한다 — 장시간 루프에서 메모리가 계속 쌓이는 걸 방지.

문서는 질문과 달리 접두사 없이 인코딩한다(1단계에서 확인한 비대칭 인코딩 규칙).

In [ ]:
import json
from pathlib import Path


def find_project_root(start_path: Path) -> Path:
    """현재 위치부터 상위 폴더를 확인해 프로젝트 루트를 찾는다."""
    start_path = start_path.resolve()
    for candidate in (start_path, *start_path.parents):
        if (candidate / ".git").exists() and (candidate / "data" / "processed").exists():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. dontalk 저장소 내부에서 노트북을 실행해 주세요.")


PROJECT_ROOT = find_project_root(Path.cwd())
RAG_PATH = PROJECT_ROOT / "data" / "processed" / "rag_documents.jsonl"
BATCH_SIZE = 32

with RAG_PATH.open(encoding="utf-8") as f:
    all_docs = [json.loads(line) for line in f]

# 이미 DB에 들어간 문서는 건너뛴다 (중간에 멈췄다 재실행해도 중복이 안 생기게)
with conn.cursor() as cur:
    cur.execute("SELECT doc_id FROM document_chunk")
    done_ids = {row[0] for row in cur.fetchall()}

docs = [d for d in all_docs if d["doc_id"] not in done_ids]
print(f"전체 {len(all_docs):,}건 중 이미 적재됨 {len(done_ids):,}건, 남은 {len(docs):,}건")

In [ ]:
import torch
from tqdm.auto import tqdm

META_FIELDS = ["category", "consulting_topic", "qa_topic", "consulting_purpose"]

def upsert_batch(cur, batch, embeddings):
    """문서 배치 + 임베딩을 document_chunk 테이블에 저장한다 (upsert라 재실행해도 중복 안 생김)."""
    rows = [
        (d["doc_id"], d["text"], *[d["metadata"].get(k) for k in META_FIELDS], emb)
        for d, emb in zip(batch, embeddings)
    ]
    cur.executemany(
        """
        INSERT INTO document_chunk (doc_id, text, category, consulting_topic, qa_topic, consulting_purpose, embedding)
        VALUES (%s, %s, %s, %s, %s, %s, %s)
        ON CONFLICT (doc_id) DO UPDATE SET
            text = EXCLUDED.text,
            category = EXCLUDED.category,
            consulting_topic = EXCLUDED.consulting_topic,
            qa_topic = EXCLUDED.qa_topic,
            consulting_purpose = EXCLUDED.consulting_purpose,
            embedding = EXCLUDED.embedding
        """,
        rows,
    )


if docs:
    with conn.cursor() as cur:
        for i in tqdm(range(0, len(docs), BATCH_SIZE), desc="임베딩+적재"):
            batch = docs[i : i + BATCH_SIZE]
            texts = [d["text"] for d in batch]
            embeddings = model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True)  # 문서: 접두사 없이 인코딩
            upsert_batch(cur, batch, embeddings)
            torch.mps.empty_cache()  # 배치마다 GPU 메모리 정리 (장시간 루프에서 메모리 누적 방지)
    print("적재 완료")
else:
    print("남은 문서가 없습니다 (이미 전부 적재됨)")

## 6단계. 적재 확인

실제로 몇 건 들어갔는지, 샘플 몇 건이 제대로 저장됐는지 확인한다.

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM document_chunk")
    print(f"document_chunk 총 {cur.fetchone()[0]:,}건")

    cur.execute("SELECT doc_id, category, consulting_topic, left(text, 60) FROM document_chunk LIMIT 3")
    for row in cur.fetchall():
        print(row)

## 7단계. HNSW 인덱스 생성

데이터를 다 넣은 뒤에 인덱스를 만든다(3단계에서 설명한 이유: 먼저 만들면 넣을 때마다 그래프를 다시 계산해서 느려짐).
`vector_cosine_ops`를 써서 코사인 거리 기준으로 인덱스를 만든다 — 임베딩할 때 코사인 유사도를 쓰기로 한 것과 맞춰야 한다.

In [ ]:
import time

t0 = time.time()
with conn.cursor() as cur:
    cur.execute("""
        CREATE INDEX IF NOT EXISTS document_chunk_embedding_idx
        ON document_chunk USING hnsw (embedding vector_cosine_ops)
    """)
print(f"HNSW 인덱스 생성 완료 ({time.time() - t0:.1f}초)")

## 8단계. 검색 테스트 — 실제 질문으로 top-k 확인

1단계에서 만든 질문 인코딩 방식(`prompt_name="query"`)을 그대로 재사용해서, 실제 질문 몇 개를 넣어보고
top-3 검색 결과가 그럴듯한지 눈으로 확인한다. `1 - (embedding <=> 질문벡터)`는 코사인 거리를 유사도 점수로 바꾼 것이다
(pgvector의 `<=>`는 거리라서 값이 작을수록 가깝다 → `1 - 거리`로 뒤집으면 클수록 유사한 점수가 된다).

In [ ]:
def search(question: str, top_k: int = 3):
    query_emb = model.encode(question, prompt_name="query", normalize_embeddings=True)
    with conn.cursor() as cur:
        cur.execute(
            """
            SELECT doc_id, category, consulting_topic, qa_topic, text,
                   1 - (embedding <=> %s) AS score
            FROM document_chunk
            ORDER BY embedding <=> %s
            LIMIT %s
            """,
            (query_emb, query_emb, top_k),
        )
        return cur.fetchall()


test_questions = [
    "대출 만기를 연장하려면 어떻게 해야 하나요?",
    "자동차 사고가 났는데 보험금 청구는 어떻게 하나요?",
    "해외 주식 매수는 어떻게 하나요?",
]

for q in test_questions:
    print(f"\n[질문] {q}")
    for doc_id, category, consulting_topic, qa_topic, text, score in search(q):
        print(f"  score={score:.4f} | {category}/{consulting_topic}/{qa_topic} | {text[:50]}...")

## 관찰 / 메모
- 1단계 확인: 임베딩 차원 1024, 질문/문서 비대칭 인코딩 정상 동작 (관련 문서 유사도 0.74 vs 무관 문서 0.18)
- **중간에 macOS가 멈춰서 강제 종료하는 사고 발생** — 처음엔 8만 건을 한 번에 처리하다 1시간 타임아웃으로 실패,
  재시작 후 다시 시도하다 시스템이 멈춤. 원인은 명확히 특정 못 했지만 MPS(맥 GPU) 장시간 사용 + 여러 프로세스 동시 실행이 의심됨
- 대응: (1) doc_id 기준으로 이미 적재된 건 건너뛰는 재개 로직 추가 (2) 배치마다 torch.mps.empty_cache() 로 GPU 메모리 정리
  (3) 단독 프로세스로만 실행 -> 이후 크래시 없이 정상 완료
- 최종 결과: 80,000건 전부 적재 (embedding NULL 0건, 중복 0건, 카테고리 분포 은행 40,000/보험 24,000/증권 16,000 원본과 일치)
- 처리 속도: 초당 약 12~15건 (MPS 기준), 8만 건 전체 기준 대략 1시간~1시간 20분 소요
- HNSW 인덱스 생성: 47.4초
- 검색 테스트 3개 질문 모두 top-3가 의미적으로 정확한 카테고리/토픽 반환 -- RAG 검색 기본 동작 확인
- (참고) "해외 주식 매수" 질문의 3위 결과가 은행 카테고리로 나왔는데, 원본 데이터 라벨링 노이즈로 보임 (내용 자체는 관련 있음)
